# Семинар: знакомство с Pandas. Задачи (решения) (5 баллов)



In [1]:
import pandas as pd
import numpy as np

### Задания для самостоятельного решения

#### Часть 1. Для датасета пассажиров Титаника

In [3]:
df = pd.read_csv("../datasets/titanic_train.csv", sep=",")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


#### 1. Какова доля семей, в которых минимальный возраст меньше 20 (семьи с детьми)?

In [8]:
#выделяем и записываем в датасет колонку с фамилиями
df["Surname"] = df["Name"].apply(lambda name: name.split(",")[0])  
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Surname
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Braund
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cumings
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Heikkinen
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Futrelle
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Allen


#### Решение (вариант 1 - условие)

In [10]:
#В таблице номера строк с фамилиями, где возраст < 20
df["Surname"][df["Age"] < 20]

7        Palsson
9         Nasser
10     Sandstrom
14       Vestrom
16          Rice
         ...    
855          Aks
869      Johnson
875        Najib
877      Petroff
887       Graham
Name: Surname, Length: 164, dtype: object

In [12]:
#Для каждой фамилии - сколько раз встречается возраст < 20
df["Surname"][df["Age"] < 20].value_counts()

Surname
Andersson    6
Goodwin      5
Panula       5
Rice         4
Skoog        4
            ..
Ryerson      1
Laroche      1
Bishop       1
Dorking      1
Graham       1
Name: count, Length: 124, dtype: int64

In [14]:
#Число уникальных значений = фамилий с возрастом < 20
df["Surname"][df["Age"] < 20].nunique()

124

In [16]:
df["Surname"].nunique()

667

In [18]:
df["Surname"].value_counts()

Surname
Andersson    9
Sage         7
Panula       6
Skoog        6
Carter       6
            ..
Hanna        1
Lewy         1
Mineff       1
Haas         1
Dooley       1
Name: count, Length: 667, dtype: int64

In [20]:
df["Surname"][df["Age"] < 20].nunique()/df["Surname"].nunique()

0.18590704647676162

#### Решение (вариант 2 - группировка) 

In [24]:
#Группируем по фамилии, для всех представителей одной фамилии - минимальный возраст (у этой фамилии) 
df.groupby("Surname")["Age"].min()  #вариант 1
#df.groupby("Surname")['Age'].agg('min') #вариант 1

Surname
Abbing           42.0
Abbott           16.0
Abelson          28.0
Adahl            30.0
Adams            26.0
                 ... 
de Mulder        30.0
de Pelsmaeker    16.0
del Carlo        29.0
van Billiard     40.5
van Melkebeke     NaN
Name: Age, Length: 667, dtype: float64

In [26]:
#Группируем по фамилии, для каждой фамилии ищем: есть ли возраст <20 (->True), суммируем 
(df.groupby("Surname")["Age"].min()<20).sum()

124

In [28]:
(df.groupby("Surname")["Age"].min()<20).sum()/df["Surname"].nunique()

0.18590704647676162

---

**Пример: `min()` и `apply(min)` дают разные результаты. Лучше применять `min()`** (в данном случае)

In [30]:
#Датасет с монетами
earnings = pd.DataFrame(
    data=[['q', 3, 4, 5],
          ['q', None, 5, 6],
          ['w', 5, 6, 7]],
    columns=['BTC', 'DOGE', 'ADA', 'ETH'],
    index=['yesterday', 'today', 'tomorrow']
)
earnings

,BTC,DOGE,ADA,ETH
yesterday,q,3.0,4,5
today,q,NaN,5,6
tomorrow,w,5.0,6,7


In [32]:
#Вариант с min()
#В группе по BTC найдем min в DOGE
earnings.groupby("BTC")['DOGE'].min()

BTC
q    3.0
w    5.0
Name: DOGE, dtype: float64

In [34]:
#В группе по BTC min<5 в DOGE - отличие (см. ниже)!!!!!!!
earnings.groupby("BTC")['DOGE'].min()<5

BTC
q     True
w    False
Name: DOGE, dtype: bool

In [36]:
#В группе по BTC min<5 в DOGE один - отличие (см. ниже)!!!!!!!
(earnings.groupby("BTC")['DOGE'].min()<5).sum() 

1

In [38]:
#Вариант с apply(min)
#В группе поBTC найдем min в DOGE
earnings.groupby("BTC")['DOGE'].apply(min)

C:\Users\ebobr\AppData\Local\Temp\ipykernel_54288\109051616.py:3: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.
  earnings.groupby("BTC")['DOGE'].apply(min)


BTC
q    NaN
w    5.0
Name: DOGE, dtype: float64

In [40]:
#В группе по по BTC min<5 в DOGE - отличие (см. выше)!!!!!!!
earnings.groupby("BTC")['DOGE'].apply(min)<5

C:\Users\ebobr\AppData\Local\Temp\ipykernel_54288\1573500625.py:2: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.
  earnings.groupby("BTC")['DOGE'].apply(min)<5


BTC
q    False
w    False
Name: DOGE, dtype: bool

In [42]:
#В группе по по BTC min<5 в DOGE один - отличие (см. выше)!!!!!!!
(earnings.groupby("BTC")['DOGE'].apply(min)<5).sum()

C:\Users\ebobr\AppData\Local\Temp\ipykernel_54288\2768630679.py:2: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.
  (earnings.groupby("BTC")['DOGE'].apply(min)<5).sum()


0

In [44]:
# Сколько семей, в которых минимальный возраст меньше 10 лет?
np.sum(df.groupby("Surname")["Age"].apply(min) < 20)

C:\Users\ebobr\AppData\Local\Temp\ipykernel_54288\3993066140.py:2: FutureWarning: The provided callable <built-in function min> is currently using np.minimum.reduce. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string np.minimum.reduce instead.
  np.sum(df.groupby("Surname")["Age"].apply(min) < 20)


119

---

#### 2. Какова доля выживших пассажиров класса 3? А пассажиров класса 1?

In [46]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Surname
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Braund
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cumings
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Heikkinen
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Futrelle
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Allen


In [48]:
(df['Survived'][df['Pclass'] == 3].sum())/len(df)

0.1335578002244669

In [50]:
(df['Survived'][df['Pclass'] == 1].sum())/len(df)

0.1526374859708193

#### 3. Сколько пассажиров выжило, а сколько - нет?

In [52]:
(df['Survived'] == 1).sum()

342

#### 4. Создайте столбец 'IsChild', который равен 1, если возраст меньше 20, и 0 - иначе. Для пропущенных значений поведение функции может быть произвольным.

In [54]:
df['IsChild'] = df["Age"].apply(lambda x: 1 if x < 20 else 0)
df.head(20)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Surname,IsChild
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Braund,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Cumings,0
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Heikkinen,0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Futrelle,0
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Allen,0
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,Moran,0
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,McCarthy,0
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S,Palsson,1
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S,Johnson,0
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C,Nasser,1


In [72]:
#смотрим на пассажира номер 17
df.iloc[17] 

PassengerId                              18
Survived                                  1
Pclass                                    2
Name           Williams, Mr. Charles Eugene
Sex                                    male
Age                                     NaN
SibSp                                     0
Parch                                     0
Ticket                               244373
Fare                                   13.0
Cabin                                   NaN
Embarked                                  S
Surname                            Williams
IsChild                                   0
Name: 17, dtype: object

In [92]:
#выведем строки, где в df['Age'] стоят пропуски
df[df['Age'].isna()].head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Surname,IsChild
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q,Moran,0
17,18,1,2,"Williams, Mr. Charles Eugene",male,NaN,0,0,244373,13.0000,NaN,S,Williams,0
19,20,1,3,"Masselmani, Mrs. Fatima",female,NaN,0,0,2649,7.2250,NaN,C,Masselmani,0
26,27,0,3,"Emir, Mr. Farred Chehab",male,NaN,0,0,2631,7.2250,NaN,C,Emir,0
28,29,1,3,"O'Dwyer, Miss. Ellen ""Nellie""",female,NaN,0,0,330959,7.8792,NaN,Q,O'Dwyer,0


In [96]:
#индексы этих строк
df.index[df['Age'].isna()]

Index([  5,  17,  19,  26,  28,  29,  31,  32,  36,  42,
       ...
       832, 837, 839, 846, 849, 859, 863, 868, 878, 888],
      dtype='int64', length=177)

#### 5. Какова доля выживших женщин из первого класса? А доля выживших мужчин из 3 класса?

In [102]:
np.sum(df['Survived'][(df['Sex'] == 'female')&(df['Pclass'] == 1)])/len(df) #(вариант 1)
#(df['Survived'][(df['Sex'] == 'female')&(df['Pclass'] == 1)]).sum()/len(df) #(вариант 2)

0.10213243546576879

In [100]:
np.sum(df['Survived'][(df['Sex'] == 'male')&(df['Pclass'] == 3)])/len(df)

0.052749719416386086

In [106]:
#Какова доля выживших женщин старше 30 лет? 
np.sum(df['Survived'][(df['Sex'] == 'female')&(df['Age'] > 30)])/len(df)

0.0931537598204265

### Задания для самостоятельного решения

#### Часть 2.

#### Описание данных

В папке Data находится информация о студентах. Всего 10 групп студентов. Файлы делятся на две категории:
    * Students_info_i - информация о студентах из группы i
    * Students_marks_i - оценки студентов из группы i за экзамены

Здесь можно посмотреть подробно о некоторых методах работы с табличными данными pandas: https://www.kaggle.com/residentmario/renaming-and-combining#Combining

#### Задача 2.1. Соберите всю информацию о студентах в одну таблицу df. В получившейся таблице должна быть информация и оценки всех студентов из всех групп. Напечатайте несколько строк таблицы для демонстрации результата.¶

In [182]:
#\ (обратный слэш):
# Используется в Windows для обозначения путей к файлам. 
# Однако обратный слэш является управляющим символом в строках Python.
#Здесь \S (см. код ниже) может быть интерпретировано как управляющая последовательность, 
# что вызовет ошибку или неправильное чтение файла.
#
#/ (прямой слэш):
#Это универсальный разделитель, который работает как в Unix-подобных 
# системах (Linux, macOS), так и в Windows. 
# В Python прямой слэш не является управляющим символом, 
# поэтому его можно использовать без дополнительных действий.

In [10]:
# file_names_info = ['Data/Students_info_0.csv', 'Data/Students_info_1.csv', 'Data/Students_info_2.csv',
#               'Data/Students_info_3.csv', 'Data/Students_info_4.csv', 'Data/Students_info_5.csv',
#               'Data/Students_info_6.csv', 'Data/Students_info_7.csv', 'Data/Students_info_8.csv',
#               'Data/Students_info_9.csv']                       

file_names_info = ['../datasets/Data/Students_info_0.csv', '../datasets/Data/Students_info_1.csv', '../datasets/Data/Students_info_2.csv',
              '../datasets/Data/Students_info_3.csv', '../datasets/Data/Students_info_4.csv', '../datasets/Data/Students_info_5.csv',
              '../datasets/Data/Students_info_6.csv', '../datasets/Data/Students_info_7.csv', '../datasets/Data/Students_info_8.csv',
              '../datasets/Data/Students_info_9.csv']          

data_info = []  # Создаем пустой список для хранения загруженных данных из файлов

for file_name in file_names_info:
    df_temp = pd.read_csv(file_name, sep=",")#, index_col=0)  # Загружаем каждый файл CSV в объект DataFrame
    data_info.append(df_temp)  # Добавляем DataFrame в список data_info

#Объединяем все загруженные данные в один DataFrame:
combined_data_info = pd.concat(data_info)
combined_data_info

,index,gender,race/ethnicity,parental level of education,lunch,test preparation course,group
0,0,female,group B,bachelor's degree,standard,none,group1
1,1,female,group C,some college,standard,completed,group1
2,2,female,group B,master's degree,standard,none,group1
3,3,male,group A,associate's degree,free/reduced,none,group1
4,4,male,group C,some college,standard,none,group1
...,...,...,...,...,...,...,...
95,995,female,group E,master's degree,standard,completed,group10
96,996,male,group C,high school,free/reduced,none,group10
97,997,female,group C,high school,free/reduced,completed,group10
98,998,female,group D,some college,standard,completed,group10


In [12]:
file_names_marks = ['../datasets/Data/Students_marks_0.csv', '../datasets/Data/Students_marks_1.csv', '../datasets/Data/Students_marks_2.csv',
              '../datasets/Data/Students_marks_3.csv', '../datasets/Data/Students_marks_4.csv', '../datasets/Data/Students_marks_5.csv',
              '../datasets/Data/Students_marks_6.csv', '../datasets/Data/Students_marks_7.csv', '../datasets/Data/Students_marks_8.csv',
              '../datasets/Data/Students_marks_9.csv']                       
data_marks = []  # Создаем пустой список для хранения загруженных данных из файлов

for file_name in file_names_marks:
    df_temp = pd.read_csv(file_name, sep=",")#, index_col=0)  # Загружаем каждый файл CSV в объект DataFrame
    data_marks.append(df_temp)  # Добавляем DataFrame в список data_marks

#Объединяем все загруженные данные в один DataFrame:
combined_data_marks = pd.concat(data_marks)
combined_data_marks

,index,math score,reading score,writing score
0,0,72,72,74
1,1,69,90,88
2,2,90,95,93
3,3,47,57,44
4,4,76,78,75
...,...,...,...,...
95,995,88,99,95
96,996,62,55,55
97,997,59,71,65
98,998,68,78,77


In [112]:
#Склеиваем две таблицы по столбцам (axis=1)
df = pd.concat([combined_data_info, combined_data_marks], axis=1)
df

,index,gender,race/ethnicity,parental level of education,lunch,test preparation course,group,index,math score,reading score,writing score
0,0,female,group B,bachelor's degree,standard,none,group1,0,72,72,74
1,1,female,group C,some college,standard,completed,group1,1,69,90,88
2,2,female,group B,master's degree,standard,none,group1,2,90,95,93
3,3,male,group A,associate's degree,free/reduced,none,group1,3,47,57,44
4,4,male,group C,some college,standard,none,group1,4,76,78,75
...,...,...,...,...,...,...,...,...,...,...,...
95,995,female,group E,master's degree,standard,completed,group10,995,88,99,95
96,996,male,group C,high school,free/reduced,none,group10,996,62,55,55
97,997,female,group C,high school,free/reduced,completed,group10,997,59,71,65
98,998,female,group D,some college,standard,completed,group10,998,68,78,77


#### Задание 2.2. Удалите столбец index у полученной таблицы. Напечатайте первые 10 строк таблицы.

In [115]:
del df['index']


In [117]:
df.head(10)

,gender,race/ethnicity,parental level of education,lunch,test preparation course,group,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,group1,72,72,74
1,female,group C,some college,standard,completed,group1,69,90,88
2,female,group B,master's degree,standard,none,group1,90,95,93
3,male,group A,associate's degree,free/reduced,none,group1,47,57,44
4,male,group C,some college,standard,none,group1,76,78,75
5,female,group B,associate's degree,standard,none,group1,71,83,78
6,female,group B,some college,standard,completed,group1,88,95,92
7,male,group B,some college,free/reduced,none,group1,40,43,39
8,male,group D,high school,free/reduced,completed,group1,64,64,67
9,female,group B,high school,free/reduced,none,group1,38,60,50


#### Задание 2.3. Выведите на экран размеры полученной таблицы

In [120]:
print(df.size)
print(df.shape)

9000
(1000, 9)


#### Задание 2.4. Выведите на экран статистические характеристики числовых столбцов таблицы (минимум, максимум, среднее значение, стандартное отклонение)

Решение (вариант 1)

In [122]:
df.describe()

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


Решение (вариант 2)

In [125]:
print('min:\n', df[['math score','reading score', 'writing score']].min())
print()
print('max:\n', df[['math score','reading score', 'writing score']].max())
print()
print('mean:\n', df[['math score','reading score', 'writing score']].mean())
print()
print('std:\n', df[['math score','reading score', 'writing score']].std())

min:
 math score        0
reading score    17
writing score    10
dtype: int64

max:
 math score       100
reading score    100
writing score    100
dtype: int64

mean:
 math score       66.089
reading score    69.169
writing score    68.054
dtype: float64

std:
 math score       15.163080
reading score    14.600192
writing score    15.195657
dtype: float64


#### Задание 2.5. Проверьте, есть ли в таблице пропущенные значения

In [128]:
#число пропусков
df.isna().sum()

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
group                          0
math score                     0
reading score                  0
writing score                  0
dtype: int64

#### Задание 2.6. Выведите на экран средние баллы студентов по каждому предмету (math, reading, writing)

In [131]:
print('mean:\n', df[['math score','reading score', 'writing score']].mean())

mean:
 math score       66.089
reading score    69.169
writing score    68.054
dtype: float64


#### Задание 2.7. Как зависят оценки от того, проходил ли студент курс для подготовки к сдаче экзамена (test preparation course)? Выведите на экран для каждого предмета в отдельности средний балл студентов, проходивших курс для подготовки к экзамену и не проходивших курс.

- __Решение (вариант 1)__

In [133]:
print(df['math score'][df['test preparation course']=='completed'].mean())
print(df['math score'][df['test preparation course']=='none'].mean())
print()
print(df['reading score'][df['test preparation course']=='completed'].mean())
print(df['reading score'][df['test preparation course']=='none'].mean())
print()
print(df['writing score'][df['test preparation course']=='completed'].mean())
print(df['writing score'][df['test preparation course']=='none'].mean())

69.69553072625699
64.0778816199377

73.89385474860335
66.53426791277259

74.41899441340782
64.50467289719626


In [135]:
#или
df[df["test preparation course"]=='completed'][['math score', 'reading score', 'writing score']].mean() 

math score       69.695531
reading score    73.893855
writing score    74.418994
dtype: float64

In [137]:
df[df["test preparation course"]=='none'][['math score', 'reading score', 'writing score']].mean() 

math score       64.077882
reading score    66.534268
writing score    64.504673
dtype: float64

- __Решение (вариант 2)__

In [139]:
#группируем по столбцу test preparation course
gr = df.groupby(by="test preparation course")
gr.nunique()


,gender,race/ethnicity,parental level of education,lunch,group,math score,reading score,writing score
test preparation course,,,,,,,,
completed,2,5,6,2,10,66,61,61
none,2,5,6,2,10,79,70,76


In [141]:
gr[['math score', 'reading score', 'writing score']].mean()

,math score,reading score,writing score
test preparation course,,,
completed,69.695531,73.893855,74.418994
none,64.077882,66.534268,64.504673


- __Решение (вариант 3)__

In [143]:
#сводная таблица
#(значение - что считаем, 
#индексы строк (горизонт. ось), 
#столбцы, 
#ф-ия для агрегации (кол-во, среднее и т.д.) - применяем к значению)
df.pivot_table(
    values=['math score', 'reading score', 'writing score'], 
    index='test preparation course', 
#    columns='math score', 
    aggfunc='mean')

,math score,reading score,writing score
test preparation course,,,
completed,69.695531,73.893855,74.418994
none,64.077882,66.534268,64.504673


#### Задание 2.8. Выведите на экран все различные значения из столбца lunch.

In [145]:
df['lunch'].value_counts()

lunch
standard        645
free/reduced    355
Name: count, dtype: int64

#### Задание 2.9. Переименуйте колонку "parental level of education" в "education", а "test preparation course" в "test preparation" с помощью метода pandas rename
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rename.html

In [148]:
df.rename(columns={'parental level of education': 'education', 'test preparation course': 'test preparation'}, inplace=True)
df

,gender,race/ethnicity,education,lunch,test preparation,group,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,group1,72,72,74
1,female,group C,some college,standard,completed,group1,69,90,88
2,female,group B,master's degree,standard,none,group1,90,95,93
3,male,group A,associate's degree,free/reduced,none,group1,47,57,44
4,male,group C,some college,standard,none,group1,76,78,75
...,...,...,...,...,...,...,...,...,...
95,female,group E,master's degree,standard,completed,group10,88,99,95
96,male,group C,high school,free/reduced,none,group10,62,55,55
97,female,group C,high school,free/reduced,completed,group10,59,71,65
98,female,group D,some college,standard,completed,group10,68,78,77


**Зафиксируем минимальный балл для сдачи экзамена**

passmark = 50

**Задание 2.10. Ответьте на вопросы:**  
    * Какая доля студентов сдала экзамен по математике (passmark >= 50)?  
    * Какая доля студентов, проходивших курс подготовки к экзамену, сдала экзамен по математике?  
    * Какая доля девушек, не проходивших курс подготовки к экзамену, не сдала экзамен по математике? 

In [153]:
passmark = 50
#Какая доля студентов сдала экзамен по математике (passmark >= 50)?  
df[df['math score'] >= passmark]

,gender,race/ethnicity,education,lunch,test preparation,group,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,group1,72,72,74
1,female,group C,some college,standard,completed,group1,69,90,88
2,female,group B,master's degree,standard,none,group1,90,95,93
4,male,group C,some college,standard,none,group1,76,78,75
5,female,group B,associate's degree,standard,none,group1,71,83,78
...,...,...,...,...,...,...,...,...,...
95,female,group E,master's degree,standard,completed,group10,88,99,95
96,male,group C,high school,free/reduced,none,group10,62,55,55
97,female,group C,high school,free/reduced,completed,group10,59,71,65
98,female,group D,some college,standard,completed,group10,68,78,77


In [155]:
len(df)

1000

In [157]:

np.sum(df['math score'] >= passmark) / len(df)

0.865

In [159]:
#в процентах
#print(f"Доля студентов, сдавших экзамен по математике: {(df['math score'] >= passmark).sum()/len(df):.2%}")

In [161]:
#или
#(df['math score'] >= passmark).mean()

In [163]:
#Какая доля студентов, проходивших курс подготовки к экзамену, сдала экзамен по математике?
#

In [165]:
np.sum((df['test preparation'] == 'completed') & (df['math score'] >= passmark)) / np.sum(df['test preparation'] == 'completed')

0.9217877094972067

In [167]:
#или
#(df[df['test preparation'] == 'completed']['math score'] >= passmark).mean()

In [169]:
#Какая доля девушек, не проходивших курс подготовки к экзамену, не сдала экзамен по математике?
#

In [171]:
np.sum((df['gender'] == 'female') & (df['test preparation'] == 'none') & (df['math score'] < passmark) ) / np.sum((df['gender'] == 'female') & (df['test preparation'] == 'none'))

0.20958083832335328

In [173]:
#или
#(df[(df['gender'] == 'female') & (df['test preparation'] == 'none')] ['math score'] < passmark).mean()

**Задание 2.11. С помощью groupby выполните задания ниже. Также выведите время выполнения каждого из заданий.**  
    * Для каждой этнической группы выведите средний балл за экзамен по чтению  
    * Для каждого уровня образования выведите минимальный балл за экзамен по письму

In [176]:
#группируем по столбцу race/ethnicity
gr = df.groupby(by="race/ethnicity")
gr.nunique()

,gender,education,lunch,test preparation,group,math score,reading score,writing score
race/ethnicity,,,,,,,,
group A,2,6,2,2,10,47,47,47
group B,2,6,2,2,10,59,58,60
group C,2,6,2,2,10,68,63,65
group D,2,6,2,2,10,61,56,60
group E,2,6,2,2,10,54,50,54


In [178]:
#Для каждой этнической группы выведите средний балл за экзамен по чтению
gr["reading score"].mean()

race/ethnicity
group A    64.674157
group B    67.352632
group C    69.103448
group D    70.030534
group E    73.028571
Name: reading score, dtype: float64

In [180]:
#группируем по столбцу education
gr = df.groupby(by="education")
gr.nunique()

,gender,race/ethnicity,lunch,test preparation,group,math score,reading score,writing score
education,,,,,,,,
associate's degree,2,5,2,2,10,61,55,61
bachelor's degree,2,5,2,2,10,54,45,47
high school,2,5,2,2,10,61,58,57
master's degree,2,5,2,2,10,39,36,35
some college,2,5,2,2,10,58,58,64
some high school,2,5,2,2,10,61,60,60


In [87]:
#%%timeit
#Для каждого уровня образования выведите минимальный балл за экзамен по письму
#группируем по столбцу education
gr = df.groupby(by="education")
#gr.nunique()
print(gr["writing score"].min())

education
associate's degree    35
bachelor's degree     38
high school           15
master's degree       46
some college          19
some high school      10
Name: writing score, dtype: int64


**Задание 2.12. Выполните задание 11 с помощью циклов. Сравните время выполнения.**

In [89]:
#%%timeit
#Возможный способ решения (Для каждого уровня образования выведите минимальный балл за экзамен по письму)
education = list(df['education'].value_counts().index)
print(education)
for ed in education:
    print(ed, (df['writing score'][df['education']==ed]).min())

['some college', "associate's degree", 'high school', 'some high school', "bachelor's degree", "master's degree"]
some college 19
associate's degree 35
high school 15
some high school 10
bachelor's degree 38
master's degree 46


**Задание 2.13. Выведите на экран средние баллы студентов по каждому предмету в зависимости от пола и уровня образования. То есть должно получиться количество групп, равных 2 * (число уровней образования), и для каждой такой группы выыведите средний балл по каждому из предметов.**

Это можно сделать с помощью сводных таблиц (pivot_table):

https://www.kaggle.com/kamilpolak/tutorial-how-to-use-pivot-table-in-pandas

In [131]:
#вариант 1 (более удобный)
#сводная таблица
#(значение - что считаем, 
#индексы строк (горизонт. ось), 
#столбцы, 
#ф-ия для агрегации (кол-во, среднее и т.д.) - применяем к значению)
df.pivot_table(
    values=['math score', 'reading score', 'writing score'], 
    index='education', 
    columns='gender', 
    aggfunc='mean')

math score            reading score             \
gender                 female       male        female       male   
education                                                           
associate's degree  65.250000  70.764151     74.120690  67.433962   
bachelor's degree   68.349206  70.581818     77.285714  68.090909   
high school         59.351064  64.705882     68.202128  61.480392   
master's degree     66.500000  74.826087     76.805556  73.130435   
some college        65.406780  69.009259     73.550847  64.990741   
some high school    59.296703  67.840909     69.109890  64.693182   

                   writing score             
gender                    female       male  
education                                    
associate's degree     74.000000  65.405660  
bachelor's degree      78.380952  67.654545  
high school            66.691489  58.539216  
master's degree        77.638889  72.608696  
some college           74.050847  63.148148  
some high school       68.285714  61.375000

In [94]:
#вариация варианта 1
df.pivot_table(
    values=['math score', 'reading score', 'writing score'], 
#   index = ['education','gender']    #или
    columns = ['education','gender'], #или
    aggfunc='mean')

education     associate's degree            bachelor's degree             \
gender                    female       male            female       male   
math score              65.25000  70.764151         68.349206  70.581818   
reading score           74.12069  67.433962         77.285714  68.090909   
writing score           74.00000  65.405660         78.380952  67.654545   

education     high school            master's degree            some college  \
gender             female       male          female       male       female   
math score      59.351064  64.705882       66.500000  74.826087    65.406780   
reading score   68.202128  61.480392       76.805556  73.130435    73.550847   
writing score   66.691489  58.539216       77.638889  72.608696    74.050847   

education                some high school             
gender              male           female       male  
math score     69.009259        59.296703  67.840909  
reading score  64.990741        69.109890  64.693182  
writing score  63.148148        68.285714  61.375000

In [96]:
#вариант 2 (менее удобный)
#сводная таблица
#(значение - что считаем, 
#индексы строк (горизонт. ось), 
#столбцы, 
#ф-ия для агрегации (кол-во, среднее и т.д.) - применяем к значению)
pd.pivot_table(df,
    values=['math score', 'reading score', 'writing score'], 
    index='gender', 
    columns='education', 
    aggfunc='mean')

math score                                                \
education associate's degree bachelor's degree high school master's degree   
gender                                                                       
female             65.250000         68.349206   59.351064       66.500000   
male               70.764151         70.581818   64.705882       74.826087   

                                             reading score                    \
education some college some high school associate's degree bachelor's degree   
gender                                                                         
female       65.406780        59.296703          74.120690         77.285714   
male         69.009259        67.840909          67.433962         68.090909   

                                                                     \
education high school master's degree some college some high school   
gender                                                                
female      68.202128       76.805556    73.550847        69.109890   
male        61.480392       73.130435    64.990741        64.693182   

               writing score                                                \
education associate's degree bachelor's degree high school master's degree   
gender                                                                       
female              74.00000         78.380952   66.691489       77.638889   
male                65.40566         67.654545   58.539216       72.608696   

                                         
education some college some high school  
gender                                   
female       74.050847        68.285714  
male         63.148148        61.375000

#### Задание 2.14. Сколько студентов успешно сдали экзамен по математике?

Создайте новый столбец в таблице df под названием Math_PassStatus и запишите в него F, если студент не сдал экзамен по математике (балл за экзамен < passmark), и P иначе.

Посчитайте количество студентов, сдавших и не сдавших экзамен по математике.

Сделайте аналогичные шаги для экзаменов по чтению и письму.

In [99]:
#Вариант 1 (apply - lambda)
df['Math_PassStatus']=df['math score'].apply(lambda x: 'P' if x>=passmark else 'F')
df

,gender,race/ethnicity,education,lunch,test preparation,group,math score,reading score,writing score,Math_PassStatus
0,female,group B,bachelor's degree,standard,none,group1,72,72,74,P
1,female,group C,some college,standard,completed,group1,69,90,88,P
2,female,group B,master's degree,standard,none,group1,90,95,93,P
3,male,group A,associate's degree,free/reduced,none,group1,47,57,44,F
4,male,group C,some college,standard,none,group1,76,78,75,P
...,...,...,...,...,...,...,...,...,...,...
95,female,group E,master's degree,standard,completed,group10,88,99,95,P
96,male,group C,high school,free/reduced,none,group10,62,55,55,P
97,female,group C,high school,free/reduced,completed,group10,59,71,65,P
98,female,group D,some college,standard,completed,group10,68,78,77,P


In [101]:
#Вариант 2 (where)
#df['Math_PassStatus'] = np.where(df['math score'] >= 50, 'P', 'F')
#df

In [103]:
(df['Math_PassStatus'] == 'P').sum()

865

In [105]:
(df['Math_PassStatus'] == 'F').sum()

135

In [107]:
#Вариант 1 (apply - lambda)
df['Reading_PassStatus']=df['reading score'].apply(lambda x: 'P' if x>=passmark else 'F')
df['Writing_PassStatus']=df['writing score'].apply(lambda x: 'P' if x>=passmark else 'F')
df

,gender,race/ethnicity,education,lunch,test preparation,group,math score,reading score,writing score,Math_PassStatus,Reading_PassStatus,Writing_PassStatus
0,female,group B,bachelor's degree,standard,none,group1,72,72,74,P,P,P
1,female,group C,some college,standard,completed,group1,69,90,88,P,P,P
2,female,group B,master's degree,standard,none,group1,90,95,93,P,P,P
3,male,group A,associate's degree,free/reduced,none,group1,47,57,44,F,P,F
4,male,group C,some college,standard,none,group1,76,78,75,P,P,P
...,...,...,...,...,...,...,...,...,...,...,...,...
95,female,group E,master's degree,standard,completed,group10,88,99,95,P,P,P
96,male,group C,high school,free/reduced,none,group10,62,55,55,P,P,P
97,female,group C,high school,free/reduced,completed,group10,59,71,65,P,P,P
98,female,group D,some college,standard,completed,group10,68,78,77,P,P,P


In [109]:
(df['Reading_PassStatus'] == 'P').sum()

910

In [111]:
(df['Reading_PassStatus'] == 'F').sum()

90

In [113]:
(df['Writing_PassStatus'] == 'P').sum()

886

In [115]:
(df['Writing_PassStatus'] == 'F').sum()

114

#### Задание 2.15. Сколько студентов успешно сдали все экзамены?

Создайте столбец OverAll_PassStatus и запишите в него для каждого студента 'F', если студент не сдал хотя бы один из трех экзаменов, а иначе 'P'.

Посчитайте количество студентов, которые сдали все экзамены.

In [117]:
#булева маска
(df['Math_PassStatus']=='P')&(df['Reading_PassStatus']=='P')&(df['Writing_PassStatus']=='P')


0      True
1      True
2      True
3     False
4      True
      ...  
95     True
96     True
97     True
98     True
99     True
Length: 1000, dtype: bool

In [119]:
df[(df['Math_PassStatus']=='P')&(df['Reading_PassStatus']=='P')&(df['Writing_PassStatus']=='P')]


,gender,race/ethnicity,education,lunch,test preparation,group,math score,reading score,writing score,Math_PassStatus,Reading_PassStatus,Writing_PassStatus
0,female,group B,bachelor's degree,standard,none,group1,72,72,74,P,P,P
1,female,group C,some college,standard,completed,group1,69,90,88,P,P,P
2,female,group B,master's degree,standard,none,group1,90,95,93,P,P,P
4,male,group C,some college,standard,none,group1,76,78,75,P,P,P
5,female,group B,associate's degree,standard,none,group1,71,83,78,P,P,P
...,...,...,...,...,...,...,...,...,...,...,...,...
95,female,group E,master's degree,standard,completed,group10,88,99,95,P,P,P
96,male,group C,high school,free/reduced,none,group10,62,55,55,P,P,P
97,female,group C,high school,free/reduced,completed,group10,59,71,65,P,P,P
98,female,group D,some college,standard,completed,group10,68,78,77,P,P,P


In [121]:
#Создаем столбец OverAll_PassStatus  (Взаимодействие между колонками) 
df['OverAll_PassStatus'] = df.apply(lambda x: 'P' if 
                                    (x['Math_PassStatus']=='P')&(x['Reading_PassStatus']=='P')&(x['Writing_PassStatus']=='P')
                                    else 'F', axis = 1)
df

,gender,race/ethnicity,education,lunch,test preparation,group,math score,reading score,writing score,Math_PassStatus,Reading_PassStatus,Writing_PassStatus,OverAll_PassStatus
0,female,group B,bachelor's degree,standard,none,group1,72,72,74,P,P,P,P
1,female,group C,some college,standard,completed,group1,69,90,88,P,P,P,P
2,female,group B,master's degree,standard,none,group1,90,95,93,P,P,P,P
3,male,group A,associate's degree,free/reduced,none,group1,47,57,44,F,P,F,F
4,male,group C,some college,standard,none,group1,76,78,75,P,P,P,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,female,group E,master's degree,standard,completed,group10,88,99,95,P,P,P,P
96,male,group C,high school,free/reduced,none,group10,62,55,55,P,P,P,P
97,female,group C,high school,free/reduced,completed,group10,59,71,65,P,P,P,P
98,female,group D,some college,standard,completed,group10,68,78,77,P,P,P,P


In [123]:
(df['OverAll_PassStatus'] == 'P').sum()

812